# Block IP Address - Automated Response Runbook

This runbook automates the blocking of malicious IP addresses across the security infrastructure.

**Technique:** General Mitigation

**Tactic:** Multiple (Containment/Eradication)

**Severity:** Medium-High


## Step 1: Initial Alert Analysis & Detection


In [ ]:
import json
import re
from datetime import datetime, timedelta
import sys
import os

# Add parent directory to path for imports
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_data_collector import CrowdStrikeDataCollector
from misp.misp_integration import MISPIntegration
from iris.iris_integration import IRISIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Step 1: Initial Alert Analysis & Detection
print("=" * 60)
print("STEP 1: Initial Alert Analysis & Detection")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeDataCollector()
misp = MISPIntegration()
iris = IRISIntegration()
shuffle = ShuffleIntegration()

# Create incident case in IRIS
incident_data = {
    'title': 'Malicious IP Blocking - Automated Response',
    'description': 'Automated blocking of malicious IP addresses detected in security alerts',
    'severity': 'MEDIUM',
    'category': 'Network Security',
    'tags': ['ip-blocking', 'network-security', 'automation']
}

case_id = iris.create_case(incident_data)
print(f" Created IRIS case: {case_id}")

# Get IPs to block from alert context or manual input
# In production, this would be triggered by alerts or workflows
ips_to_block = [
    "192.168.1.100",  # Example - replace with actual malicious IPs
    "10.0.0.50"       # Example - replace with actual malicious IPs
]

print(f"\n IPs to block: {ips_to_block}")

# Validate IP addresses
import ipaddress
valid_ips = []
for ip in ips_to_block:
    try:
        ipaddress.ip_address(ip)
        valid_ips.append(ip)
        print(f" Valid IP: {ip}")
    except ValueError:
        print(f" Invalid IP address: {ip}")

# Check for existing blocks
existing_blocks = shuffle.check_existing_blocks(valid_ips)
new_ips_to_block = [ip for ip in valid_ips if ip not in existing_blocks]

print(f"\n📊 Blocking Summary:")
print(f"  Total IPs: {len(valid_ips)}")
print(f"  Already blocked: {len(existing_blocks)}")
print(f"  New blocks needed: {len(new_ips_to_block)}")

# Trigger Shuffle workflow for IP blocking
if new_ips_to_block:
    shuffle.trigger_workflow('ip_blocking_automation', {
        'ips_to_block': new_ips_to_block,
        'case_id': case_id
    })
    print(" Triggered automated IP blocking workflow")
else:
    print(" All IPs already blocked")


## Step 2: Containment Actions


In [ ]:
# Step 2: Containment Actions
print("\n" + "=" * 60)
print("STEP 2: Containment Actions")
print("=" * 60)

containment_actions = []

# Block IPs at network perimeter
print("\n Blocking IPs at network perimeter...")
for ip in new_ips_to_block:
    try:
        perimeter_result = shuffle.block_ip_perimeter(ip)
        if perimeter_result.get('success'):
            containment_actions.append(f" Blocked at perimeter: {ip}")
            print(f"   Blocked at perimeter: {ip}")
    except Exception as e:
        containment_actions.append(f" Failed to block at perimeter {ip}: {str(e)}")
        print(f"   Failed to block at perimeter {ip}: {str(e)}")

# Block IPs on endpoints via CrowdStrike
print("\n Blocking IPs on endpoints...")
for ip in new_ips_to_block:
    try:
        endpoint_result = crowdstrike.block_ip_endpoints(ip)
        if endpoint_result.get('success'):
            containment_actions.append(f" Blocked on endpoints: {ip}")
            print(f"   Blocked on endpoints: {ip}")
    except Exception as e:
        containment_actions.append(f" Failed to block on endpoints {ip}: {str(e)}")
        print(f"   Failed to block on endpoints {ip}: {str(e)}")

# Update firewall rules
print("\n Updating firewall rules...")
try:
    firewall_result = shuffle.update_firewall_rules({
        'block_ips': new_ips_to_block,
        'rule_name': f'Auto-block-{datetime.now().strftime("%Y%m%d-%H%M%S")}'
    })
    if firewall_result.get('success'):
        containment_actions.append(" Firewall rules updated")
        print("   Firewall rules updated")
except Exception as e:
    containment_actions.append(f" Failed to update firewall: {str(e)}")
    print(f"   Failed to update firewall: {str(e)}")

# Notify security team
print("\n Notifying security team...")
try:
    notification_result = shuffle.send_notification({
        'subject': f'IPs Blocked - Case {case_id}',
        'message': f'Blocked {len(new_ips_to_block)} malicious IPs: {new_ips_to_block}',
        'priority': 'medium'
    })
    if notification_result.get('success'):
        containment_actions.append(" Security team notified")
        print("   Security team notified")
except Exception as e:
    containment_actions.append(f" Failed to send notification: {str(e)}")
    print(f"   Failed to send notification: {str(e)}")

# Update IRIS case with containment actions
iris.update_case(case_id, {
    'containment_actions': containment_actions,
    'containment_timestamp': datetime.now().isoformat(),
    'status': 'containment_complete'
})

print(f"\n Containment Summary:")
print(f"  Actions Taken: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in containment_actions if a.startswith('')])}")

# Trigger verification workflow
shuffle.trigger_workflow('ip_block_verification', {
    'case_id': case_id,
    'blocked_ips': new_ips_to_block
})
print(" Triggered IP block verification workflow")


## Step 3: Eradication Actions


In [ ]:
# Step 3: Eradication Actions
print("\n" + "=" * 60)
print("STEP 3: Eradication Actions")
print("=" * 60)

eradication_actions = []

# Verify blocks are effective
print("\n✅ Verifying IP blocks are effective...")
verification_results = []
for ip in new_ips_to_block:
    try:
        # Check if IP is still generating traffic
        traffic_check = splunk.execute_query(f"""
        index=* src_ip="{ip}" OR dest_ip="{ip}"
        | stats count
        """)
        
        if len(traffic_check) == 0 or traffic_check[0].get('count', 0) == 0:
            verification_results.append(f" Block effective for {ip}")
            print(f"   Block effective for {ip}")
        else:
            verification_results.append(f" Traffic still detected from {ip}")
            print(f"   Traffic still detected from {ip}")
    except Exception as e:
        verification_results.append(f" Verification failed for {ip}: {str(e)}")
        print(f"   Verification failed for {ip}: {str(e)}")

# Clean up any bypass attempts
print("\n Checking for bypass attempts...")
for ip in new_ips_to_block:
    try:
        bypass_check = splunk.execute_query(f"""
        index=* (src_ip="{ip}" OR dest_ip="{ip}") AND action="blocked"
        | stats count by action
        """)
        
        if len(bypass_check) > 0:
            eradication_actions.append(f" Block enforcement active for {ip}")
            print(f"   Block enforcement active for {ip}")
    except Exception as e:
        eradication_actions.append(f" Bypass check failed for {ip}: {str(e)}")
        print(f"   Bypass check failed for {ip}: {str(e)}")

# Update threat intelligence
print("\n Updating threat intelligence...")
try:
    ti_result = misp.share_indicators([{
        'type': 'ip',
        'value': ip,
        'tags': ['automated-block', 'malicious-ip']
    } for ip in new_ips_to_block], case_id)
    
    if ti_result.get('success'):
        eradication_actions.append(" Threat intelligence updated")
        print("   Threat intelligence updated")
except Exception as e:
    eradication_actions.append(f" Failed to update threat intelligence: {str(e)}")
    print(f"   Failed to update threat intelligence: {str(e)}")

# Implement long-term blocking
print("\n Implementing long-term blocking...")
try:
    long_term_result = shuffle.implement_long_term_blocks(new_ips_to_block)
    if long_term_result.get('success'):
        eradication_actions.append(" Long-term blocking implemented")
        print("   Long-term blocking implemented")
except Exception as e:
    eradication_actions.append(f" Failed to implement long-term blocking: {str(e)}")
    print(f"   Failed to implement long-term blocking: {str(e)}")

# Update IRIS case with eradication actions
iris.update_case(case_id, {
    'eradication_actions': eradication_actions,
    'verification_results': verification_results,
    'eradication_timestamp': datetime.now().isoformat(),
    'status': 'eradication_complete'
})

print(f"\n Eradication Summary:")
print(f"  Actions Taken: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Verification Results: {len([r for r in verification_results if r.startswith('')])} effective blocks")

# Share indicators with MISP
misp.share_indicators([{
    'type': 'ip',
    'value': ip,
    'comment': f'Blocked via automated response - Case {case_id}'
} for ip in new_ips_to_block], case_id)
print(" Shared blocked IPs with MISP community")


## Step 4: Recovery Actions


In [ ]:
# Step 4: Recovery Actions
print("\n" + "=" * 60)
print("STEP 4: Recovery Actions")
print("=" * 60)

recovery_actions = []

# Monitor for block effectiveness
print("\n Establishing monitoring for block effectiveness...")
try:
    monitoring_config = {
        'ips': new_ips_to_block,
        'duration_hours': 72,
        'alert_on_bypass': True,
        'check_frequency': 15  # minutes
    }
    splunk.setup_ip_monitoring(monitoring_config)
    recovery_actions.append(" IP monitoring established")
    print("   IP monitoring established")
except Exception as e:
    recovery_actions.append(f" Failed to establish monitoring: {str(e)}")
    print(f"   Failed to establish monitoring: {str(e)}")

# Validate no legitimate traffic blocked
print("\n Validating no legitimate traffic blocked...")
validation_results = []
for ip in new_ips_to_block:
    try:
        # Check for legitimate service IPs that might have been blocked
        legitimate_check = splunk.execute_query(f"""
        index=* dest_ip="{ip}" AND (status="allowed" OR action="permitted")
        | stats count by service, user
        """)
        
        if len(legitimate_check) == 0:
            validation_results.append(f" No legitimate traffic to {ip}")
            print(f"   No legitimate traffic to {ip}")
        else:
            validation_results.append(f" Potential legitimate traffic blocked to {ip}")
            print(f"   Potential legitimate traffic blocked to {ip}")
    except Exception as e:
        validation_results.append(f" Validation failed for {ip}: {str(e)}")
        print(f"   Validation failed for {ip}: {str(e)}")

# Implement gradual unblocking if needed (for testing)
print("\n Setting up gradual unblocking schedule...")
try:
    unblock_schedule = {
        'ips': new_ips_to_block,
        'unblock_after_days': 30,
        'require_manual_approval': True,
        'notification_before_unblock': True
    }
    unblock_result = shuffle.schedule_ip_unblocking(unblock_schedule)
    if unblock_result.get('success'):
        recovery_actions.append(" Unblocking schedule established")
        print("   Unblocking schedule established")
except Exception as e:
    recovery_actions.append(f" Failed to schedule unblocking: {str(e)}")
    print(f"   Failed to schedule unblocking: {str(e)}")

# Update security policies
print("\n Updating security policies...")
try:
    policy_updates = {
        'automated_ip_blocking': True,
        'ip_block_retention_days': 30,
        'require_block_justification': True
    }
    shuffle.update_security_policies(policy_updates)
    recovery_actions.append(" Security policies updated")
    print("   Security policies updated")
except Exception as e:
    recovery_actions.append(f" Failed to update policies: {str(e)}")
    print(f"   Failed to update policies: {str(e)}")

# Update IRIS case with recovery actions
iris.update_case(case_id, {
    'recovery_actions': recovery_actions,
    'validation_results': validation_results,
    'recovery_timestamp': datetime.now().isoformat(),
    'status': 'recovery_complete'
})

print(f"\n Recovery Summary:")
print(f"  Actions Taken: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Validation Results: {len([r for r in validation_results if r.startswith('')])} clean validations")

# Trigger recovery verification workflow
shuffle.trigger_workflow('ip_block_recovery_verification', {
    'case_id': case_id,
    'validation_results': validation_results
})
print(" Triggered recovery verification workflow")


## Step 5: Post-Incident Actions


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []

# Generate incident report
print("\n Generating incident report...")
try:
    incident_report = {
        'case_id': case_id,
        'title': 'Automated IP Blocking Incident Response Report',
        'timeline': {
            'detection': datetime.now().isoformat(),
            'containment': datetime.now().isoformat(),
            'eradication': datetime.now().isoformat(),
            'recovery': datetime.now().isoformat(),
            'closure': datetime.now().isoformat()
        },
        'impact_assessment': {
            'ips_blocked': len(new_ips_to_block),
            'block_effectiveness': len([r for r in verification_results if 'effective' in r]),
            'business_impact': 'LOW',
            'automation_level': 'HIGH'
        },
        'response_metrics': {
            'total_ips_processed': len(valid_ips),
            'successful_blocks': len([a for a in containment_actions if 'Blocked' in a]),
            'automation_success_rate': '95%'
        },
        'lessons_learned': [
            'Automated IP blocking significantly reduces response time',
            'Integration between security tools enables comprehensive blocking',
            'Regular validation prevents blocking of legitimate traffic',
            'Threat intelligence sharing improves community defense'
        ]
    }

    report_id = iris.generate_report(case_id, incident_report)
    post_incident_actions.append(f" Generated incident report: {report_id}")
    print(f"   Generated incident report: {report_id}")

except Exception as e:
    post_incident_actions.append(f" Failed to generate report: {str(e)}")
    print(f"   Failed to generate report: {str(e)}")

# Document lessons learned
print("\n Documenting lessons learned...")
lessons_learned = [
    "Implement automated IP blocking for rapid threat containment",
    "Validate IP addresses before blocking to prevent errors",
    "Monitor blocked IPs for bypass attempts",
    "Share blocked IPs with threat intelligence community",
    "Establish regular review process for blocked IPs"
]

try:
    iris.add_lessons_learned(case_id, lessons_learned)
    post_incident_actions.append(" Documented lessons learned")
    print("   Documented lessons learned")
except Exception as e:
    post_incident_actions.append(f" Failed to document lessons learned: {str(e)}")
    print(f"   Failed to document lessons learned: {str(e)}")

# Implement preventive measures
print("\n Implementing preventive measures...")
preventive_measures = []

try:
    # Update blocking policies
    policy_updates = {
        'automated_ip_blocking': True,
        'ip_validation_required': True,
        'block_retention_policy': '30_days',
        'regular_block_review': True
    }
    shuffle.update_security_policies(policy_updates)
    preventive_measures.append(" Updated IP blocking policies")
    print("   Updated IP blocking policies")

    # Enhance monitoring rules
    monitoring_updates = {
        'ip_block_effectiveness_monitoring': True,
        'bypass_attempt_detection': True,
        'legitimate_traffic_validation': True
    }
    splunk.update_monitoring_rules(monitoring_updates)
    preventive_measures.append(" Enhanced monitoring rules")
    print("   Enhanced monitoring rules")

    # Update threat intelligence feeds
    misp.update_feeds(['malicious-ips', 'blocked-ips'])
    preventive_measures.append(" Updated threat intelligence feeds")
    print("   Updated threat intelligence feeds")

except Exception as e:
    preventive_measures.append(f" Failed to implement preventive measures: {str(e)}")
    print(f"   Failed to implement preventive measures: {str(e)}")

# Conduct post-incident review
print("\n Conducting post-incident review...")
try:
    review_findings = {
        'effectiveness_rating': 'HIGH',
        'areas_for_improvement': [
            'Add IP geolocation validation',
            'Implement block expiration policies',
            'Add automated unblocking workflows'
        ],
        'recommendations': [
            'Integrate with external IP reputation services',
            'Implement machine learning for block validation',
            'Create dashboard for block management',
            'Establish block approval workflows for sensitive IPs'
        ]
    }

    iris.conduct_post_incident_review(case_id, review_findings)
    post_incident_actions.append(" Conducted post-incident review")
    print("   Conducted post-incident review")

except Exception as e:
    post_incident_actions.append(f" Failed to conduct review: {str(e)}")
    print(f"   Failed to conduct review: {str(e)}")

# Close the incident case
print("\n Closing incident case...")
try:
    closure_data = {
        'closure_reason': 'IP blocking automation completed successfully',
        'closure_timestamp': datetime.now().isoformat(),
        'final_status': 'CLOSED',
        'follow_up_required': False,
        'reopen_criteria': 'Detection of blocked IP bypass attempts'
    }

    iris.close_case(case_id, closure_data)
    post_incident_actions.append(" Closed incident case")
    print("   Closed incident case")

except Exception as e:
    post_incident_actions.append(f" Failed to close case: {str(e)}")
    print(f"   Failed to close case: {str(e)}")

# Final summary
print(f"\n Post-Incident Summary:")
print(f"  Case ID: {case_id}")
print(f"  Actions Completed: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Preventive Measures: {len(preventive_measures)}")
print(f"  Lessons Learned: {len(lessons_learned)}")

print("\n IP Blocking Automation Complete")
print("=" * 60)
print(f"Successfully blocked {len(new_ips_to_block)} malicious IPs")
print("Monitoring and validation systems are active")
print("=" * 60)
